# 12 · Modelo final

**Propósito:** entrenar el modelo definitivo de la alerta temprana y elegirlo por lo que **le aporta a LATAM en
dólares**, no por una métrica estadística. La decisión de negocio es **reprogramar con anticipación** (a T−2 h) los
vuelos alertados. El modelo y su umbral se eligen por el **beneficio neto en US$** en valid (LATAM Brasil):

`beneficio = costo de demora sorpresa que se evita − costo de los minutos anunciados como demora planificada`

Los dos costos vienen de University of Westminster (2015) para EUROCONTROL, ponderados por la flota de LATAM (ver
"Costos de la demora"). Son dos alertas:

| Alerta | Objetivo | Qué avisa |
|---|---|---|
| Ámbar | `y_demora15` | la salida tendrá 15 min o más de demora |
| Roja | `y_demora60` | la salida tendrá 60 min o más de demora |

- **Modelos:** Random Forest y GBT (árboles), comparados con la regresión logística del notebook 10.
- **Referencias:** "nunca alertar" (lo que pasa hoy sin sistema), la tasa histórica por aeropuerto y hora (una tabla
  dinámica, sin Big Data ni ML) y la regresión logística base.
- **Partición temporal** (nunca al azar): se entrena con train, se eligen hiperparámetros y umbral con valid y se
  informa en test (2025), que no se mira hasta el final.
- **Veredicto:** el modelo solo se recomienda si en test deja **más dólares que la tasa histórica** y más que no
  hacer nada. Si no lo logra, el notebook lo dice.
- El modelo se entrena **por lotes** con toda la historia y se guarda como `PipelineModel`: los notebooks 13 (lote
  mensual) y 17 (capa de velocidad) lo **aplican** con `transform`, nunca lo reentrenan.

**Entrada:** `gold/features_salida` (nunca `scale-test/`) y `gold/modelos/modelo_base`.

**Salida:** `gold/modelos/final_y_demora15`, `gold/modelos/final_y_demora60`, `gold/modelos/umbrales_final` y
evidencias en `docs/evidencias/modelo/`.

**Requisito:** notebooks 08 y 10.

In [ ]:
import os
from pathlib import Path

import yaml

# Carpeta del repositorio (este notebook está en notebooks/)
RAIZ = Path.cwd().parent
with open(RAIZ / "config" / "pipeline.yaml", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)
tablas = cfg["lake"]["tablas"]

# En la EC2 la variable LAKE_BUCKET trae el nombre del bucket; en una laptop se trabaja sobre data/lake
BUCKET = os.environ.get("LAKE_BUCKET", "").strip()
RAIZ_LOCAL = RAIZ / cfg["lake"]["raiz_local"]
BASE = f"s3a://{BUCKET}" if BUCKET else RAIZ_LOCAL.resolve().as_uri()


def ruta(tabla: str) -> str:
    """Ruta completa de una tabla declarada en config/pipeline.yaml."""
    return f"{BASE}/{tablas[tabla]}"


def meses(desde: str, hasta: str) -> list[tuple[int, int]]:
    """Pares (año, mes) entre dos fechas AAAA-MM, ambas incluidas."""
    anio, mes = map(int, desde.split("-"))
    anio_fin, mes_fin = map(int, hasta.split("-"))
    salida = []
    while (anio, mes) <= (anio_fin, mes_fin):
        salida.append((anio, mes))
        anio, mes = (anio + 1, 1) if mes == 12 else (anio, mes + 1)
    return salida


DESDE, HASTA = cfg["periodo"]["inicio"], cfg["periodo"]["fin"]
print("Destino:", f"s3://{BUCKET}" if BUCKET else "data/lake (local)")
print("Periodo:", DESDE, "a", HASTA)

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import itertools
import json
from datetime import UTC, datetime

import matplotlib.pyplot as plt
import pandas as pd
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.classification import GBTClassifier, RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, RegressionEvaluator
from pyspark.ml.feature import Imputer, StringIndexer, VectorAssembler
from pyspark.ml.functions import vector_to_array
from pyspark.ml.regression import GBTRegressor

# En la EC2 Spark toma la configuración del clúster (spark-defaults.conf); en una laptop corre en modo local
constructor = SparkSession.builder.appName("latam-bigdata-modelo-final")
if not BUCKET:
    constructor = (
        constructor.master("local[*]")
        .config("spark.sql.shuffle.partitions", str(cfg["spark"]["particiones_shuffle_local"]))
        .config("spark.driver.memory", os.environ.get("SPARK_DRIVER_MEMORY", "3g"))
    )
spark = (
    constructor.config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.sources.partitionOverwriteMode", "dynamic")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")


def en_periodo(df, col_anio="year", col_mes="month"):
    """Deja solo las filas cuyo año y mes están dentro del periodo de config/pipeline.yaml."""
    ym = F.format_string("%04d-%02d", F.col(col_anio), F.col(col_mes))
    return df.where((ym >= DESDE) & (ym <= HASTA))

In [ ]:
EVID = RAIZ / "docs" / "evidencias" / "modelo"
EVID.mkdir(parents=True, exist_ok=True)
assert "scale-test" not in ruta("gold_features")

MF = cfg["modelo_final"]
OBJETIVOS = MF["objetivos"]
NUMERICAS = list(cfg["features"]["numericas"])
CATEGORICAS = list(cfg["features"]["categoricas"])
CAT_COLS = [f"cat_{c}" for c in CATEGORICAS]
VERSION = datetime.now(UTC).strftime("%Y%m%dT%H%M%S")
MODELOS_DIR = f"{BASE}/{tablas['gold_modelos']}"


def preparar(df):
    """Columnas que usa el modelo: numéricas como double y categóricas como texto (las vacías, 'desconocido')."""
    return df.select(
        "id_vuelo", "particion", "pais", "es_latam", "aerolinea", "origen", "fecha_local", "year", "month",
        "y_demora15", "y_demora60", "demora_salida_min",
        *[F.col(c).cast("double").alias(c) for c in NUMERICAS],
        *[F.coalesce(F.col(c).cast("string"), F.lit("desconocido")).alias(f"cat_{c}") for c in CATEGORICAS],
    )


datos = preparar(spark.read.parquet(ruta("gold_features"))).where(F.col("y_demora15").isNotNull())
ES_LATAM_BR = F.col("es_latam") & (F.col("pais") == "BR")
print("Versión de esta corrida:", VERSION)

## Costos de la demora (todos con fuente)

La decisión que se evalúa es la de tu planteamiento: **¿conviene reprogramar con anticipación?** A T−2 h el CCO
anuncia para el vuelo alertado un nuevo horario `m` minutos más tarde (15 min para la ámbar y 60 min para la roja).

| Pieza | Valor | Fuente |
|---|---|---|
| Costo de una demora **sorpresa** de *d* minutos (tripulación, avión, pasajeros y efecto en cadena) | Tabla 26 (base), H1 (bajo) y H2 (alto), en puerta | University of Westminster (2015), *European airline delay cost reference values* v4.1, EUROCONTROL, págs. 11 y 20 |
| Costo de un minuto de demora **planificada** (*"anticipated and accounted for in advance"*), sin costo de pasajeros ni efecto en cadena | Tabla 9 | ídem, págs. 7 y 40 |
| Avión | A319, A320 y A321 ponderados por la flota de corto alcance al 31-dic-2025 | LATAM Airlines Group, Form 20-F 2025, págs. 41 y 55 |
| Euros de 2014 → dólares del año de test | inflación de la zona euro y tipo de cambio promedio anual | Eurostat `prc_hicp_aind` y BCE `EXR` |

Para un vuelo alertado con demora real *d*:

- **Costo evitable** = sorpresa(*d*) − sorpresa(max(*d* − *m*, 0)). Los primeros *m* minutos dejan de ser sorpresa.
- **Costo de reprogramar** = *m* × costo del minuto planificado. Se paga siempre, también en las falsas alarmas.

Westminster mide los dos extremos: una demora totalmente sorpresa y otra planificada con meses de anticipación.
Avisar 2 h antes queda en medio, y **ninguna fuente publica qué fracción *f* de la diferencia se captura**. Por eso
*f* no se fija:

- el costo evitable con *f* = 1 es la **cota superior**;
- la cifra principal es el **punto de equilibrio**: *f\** = costo de reprogramar ÷ costo evitable, es decir, la
  fracción mínima que hay que capturar para que reprogramar convenga.

In [ ]:
# Costo de una demora en puerta (University of Westminster, 2015, v4.1, EUROCONTROL), ponderado por la flota de
# corto alcance de LATAM (Form 20-F 2025) y llevado a dólares del año de test con Eurostat (HICP) y el BCE.
VN = cfg["valor_negocio"]
_cw, _conv = VN["costos_westminster"], VN["conversion"]
ANIO_PRECIOS = int(_conv["anio_precios"])
FACTOR_USD = _conv["hicp_eurostat"][ANIO_PRECIOS] / _conv["hicp_eurostat"][2014] * _conv["usd_por_eur_bce"][ANIO_PRECIOS]
_flota = VN["flota_corto_alcance"]
PESO_AVION = {a: n / sum(_flota.values()) for a, n in _flota.items()}
MINUTOS_TABLA = [0] + [int(m) for m in _cw["minutos"]]
COSTOS = {}
for _esc in VN["escenarios"]:
    _tactica = [sum(PESO_AVION[a] * _cw["aviones"][a]["tactica_eur"][_esc][i] for a in PESO_AVION) * FACTOR_USD
                for i in range(len(_cw["minutos"]))]
    _estrategica = sum(PESO_AVION[a] * _cw["aviones"][a]["estrategica_eur_hora"][_esc] for a in PESO_AVION) / 60
    COSTOS[_esc] = {"tactica_usd": [0.0] + _tactica, "planificada_usd_min": _estrategica * FACTOR_USD}
M_REPROG = {p: int(m) for p, m in VN["minutos_reprogramacion"].items()}


def costo_sorpresa(col, esc):
    """US$ que cuesta una demora no planificada de `col` minutos (interpolación lineal de la tabla de Westminster).

    Pasados los 300 min (fin de la tabla) se usa el valor de 300 min, así que es una cota inferior.
    """
    d = F.least(F.greatest(col.cast("double"), F.lit(0.0)), F.lit(float(MINUTOS_TABLA[-1])))
    y = COSTOS[esc]["tactica_usd"]
    expr = F.lit(float(y[-1]))
    for i in range(len(MINUTOS_TABLA) - 1, 0, -1):
        x0, x1 = MINUTOS_TABLA[i - 1], MINUTOS_TABLA[i]
        pendiente = (y[i] - y[i - 1]) / (x1 - x0)
        expr = F.when(d <= x1, F.lit(float(y[i - 1])) + (d - F.lit(float(x0))) * F.lit(pendiente)).otherwise(expr)
    return expr


def costo_evitable(col, producto, esc):
    """Cuánto del costo sorpresa desaparece si se reprograma con anticipación: los primeros m minutos pasan a ser
    demora planificada y solo el resto (demora - m) sigue siendo sorpresa. Cota superior (fracción capturada = 1)."""
    m = float(M_REPROG[producto])
    resto = F.greatest(col.cast("double") - F.lit(m), F.lit(0.0))
    return costo_sorpresa(col, esc) - costo_sorpresa(resto, esc)


def costo_reprogramar(producto, esc):
    """US$ de los m minutos que se anuncian como demora planificada (se paga aunque el vuelo no se fuera a demorar)."""
    return COSTOS[esc]["planificada_usd_min"] * M_REPROG[producto]


print(f"Factor € 2014 -> US$ {ANIO_PRECIOS}: {FACTOR_USD:.4f} · peso de cada avión:",
      {a: round(p, 3) for a, p in PESO_AVION.items()})
pd.DataFrame([{"escenario": e, "planificada_usd_min": round(c["planificada_usd_min"], 2),
               **{f"sorpresa_{m}min_usd": round(v) for m, v in zip(MINUTOS_TABLA[1:], c["tactica_usd"][1:])},
               **{f"reprogramar_{p}_usd": round(costo_reprogramar(p, e), 2) for p in M_REPROG}}
              for e, c in COSTOS.items()])

In [ ]:
# Con qué se elige el modelo y el umbral: beneficio en US$ en valid (LATAM Brasil)
ESC = MF["escenario_seleccion"]
F_SEL = float(MF["fraccion_seleccion"])
FRACCIONES = [float(f) for f in VN["fracciones_captura"]]
PRODUCTO = {"y_demora15": "ambar", "y_demora60": "rojo"}
TRAMOS = 200
NUNCA = 1.01  # umbral que no alerta nunca: se publica si ningún umbral deja beneficio positivo


def con_evitable(df, esc=ESC):
    """Agrega el costo evitable de cada vuelo si se lo reprograma (una columna por alerta)."""
    for objetivo, producto in PRODUCTO.items():
        df = df.withColumn(f"evit_{objetivo}", costo_evitable(F.col("demora_salida_min"), producto, esc))
    return df


def curva(df, objetivo, score):
    """Por cada umbral i/200 (una sola pasada): alertas, aciertos y costo evitable (se alerta si score >= umbral)."""
    t = (
        df.withColumn("tramo", F.least(F.floor(F.col(score) * TRAMOS), F.lit(TRAMOS - 1)))
        .groupBy("tramo")
        .agg(F.sum(objetivo).alias("pos"), F.count("*").alias("n"), F.sum(f"evit_{objetivo}").alias("evit"))
        .toPandas().set_index("tramo").reindex(range(TRAMOS), fill_value=0).sort_index(ascending=False)
    )
    t["vp"], t["alertas"], t["evit"] = t.pos.cumsum(), t.n.cumsum(), t.evit.cumsum()
    t["fp"], t["fn"] = t.alertas - t.vp, t.pos.sum() - t.vp
    t["umbral"] = t.index / TRAMOS
    return t


def beneficio_usd(c, objetivo, f, esc=ESC):
    """Beneficio neto de reprogramar las alertas de cada umbral: f × costo evitable − costo de reprogramar."""
    return f * c.evit - c.alertas * costo_reprogramar(PRODUCTO[objetivo], esc)


def umbral_por_valor(c, objetivo, f=F_SEL, esc=ESC):
    """Umbral que deja más dólares. Si ninguno supera a no reprogramar nada (0 US$), no se alerta."""
    b = beneficio_usd(c, objetivo, f, esc)
    if b.max() <= 0:
        return NUNCA, 0.0
    i = b.idxmax()
    return float(c.loc[i, "umbral"]), float(b.loc[i])


print(f"Se elige con el escenario '{ESC}' de Westminster y fracción capturada {F_SEL} (cota superior)")

## 1. Train y valid para la búsqueda

Brasil entra completo; de EE. UU. se toma la fracción de `modelo_final.muestra_train` (por defecto 25 %), con
semilla fija. EE. UU. aporta volumen para aprender patrones generales (hora, clima, red), pero con todo su volumen
cada árbol tardaría varias veces más en un clúster de 2 nodos de 8 GiB, sin cambiar lo que importa: la evaluación en
LATAM Brasil. Train y valid se guardan en memoria (`cache`) antes de la búsqueda porque se recorren muchas veces
(optimización O3 del notebook 11).

In [ ]:
fr_train = {k: float(v) for k, v in MF["muestra_train"].items()}
fr_valid = {k: float(v) for k, v in MF["muestra_valid"].items()}
train_total = datos.where(F.col("particion") == "train")
train = train_total.sampleBy("pais", fr_train, seed=MF["semilla"]).cache()
valid = datos.where(F.col("particion") == "valid").sampleBy("pais", fr_valid, seed=MF["semilla"]).cache()
test = datos.where(F.col("particion") == "test")

conteo = (
    train.withColumn("conjunto", F.lit("train")).unionByName(valid.withColumn("conjunto", F.lit("valid")))
    .groupBy("conjunto", "pais").agg(F.count("*").alias("vuelos"), F.round(F.avg("y_demora15"), 4).alias("tasa15"),
                                     F.round(F.avg("y_demora60"), 4).alias("tasa60"))
    .orderBy("conjunto", "pais").toPandas()
)
conteo

## 2. Preparación común y búsqueda de hiperparámetros

- Las categóricas pasan por `StringIndexer` (los árboles no necesitan *one-hot*: cortan por categoría). Una categoría
  nueva en valid o test va a un índice propio (`handleInvalid="keep"`).
- Las numéricas faltantes se completan con la **mediana de train** (`Imputer`). Se ajusta una sola vez con train y se
  aplica igual a valid, test y al flujo en tiempo real.
- Cada vuelo demorado pesa `vuelos a tiempo / vuelos demorados` (clases desbalanceadas).
- Se elige la combinación que deja **más dólares en valid para LATAM Brasil** (la población del negocio), cada una
  con su mejor umbral. El AUC-ROC y el AUC-PR (calculados sobre `probability`, la probabilidad de la clase 1) se
  informan como diagnóstico y solo desempatan.

In [ ]:
con_dato = train.select([F.count(c).alias(c) for c in NUMERICAS]).first().asDict()
NUM_USADAS = [c for c in NUMERICAS if con_dato[c] > 0]
print("Variables numéricas sin datos en train (se omiten):", [c for c in NUMERICAS if c not in NUM_USADAS])

preparacion = Pipeline(stages=[
    StringIndexer(inputCols=CAT_COLS, outputCols=[f"{c}_idx" for c in CAT_COLS], handleInvalid="keep"),
    Imputer(inputCols=NUM_USADAS, outputCols=[f"{c}_imp" for c in NUM_USADAS], strategy="median"),
    VectorAssembler(inputCols=[f"{c}_imp" for c in NUM_USADAS] + [f"{c}_idx" for c in CAT_COLS],
                    outputCol="vector", handleInvalid="keep"),
]).fit(train)
NOMBRES_VECTOR = NUM_USADAS + CAT_COLS

# maxBins debe superar el número de categorías de la variable con más valores (aerolíneas)
cardinalidad = max(len(etiquetas) for etiquetas in preparacion.stages[0].labelsArray) + 1
MAX_BINS = max(int(MF["max_bins"]), cardinalidad + 1)
print("Categorías por variable:", dict(zip(CAT_COLS, [len(e) for e in preparacion.stages[0].labelsArray])),
      "· maxBins:", MAX_BINS)

train_v = preparacion.transform(train).cache()
valid_v = preparacion.transform(valid).cache()
print(f"train: {train_v.count():,} vuelos · valid: {valid_v.count():,} vuelos")

In [ ]:
def con_peso(df, objetivo):
    """Agrega la columna peso para balancear las clases."""
    fila = df.agg(F.sum(objetivo).alias("pos"), F.count("*").alias("n")).first()
    peso = (fila.n - fila.pos) / max(fila.pos, 1)
    return df.withColumn("peso", F.when(F.col(objetivo) == 1, F.lit(peso)).otherwise(F.lit(1.0))), peso


def con_score(df, nombre="score"):
    """Probabilidad de la clase 1 (demora) como número."""
    return df.withColumn(nombre, vector_to_array("probability")[1])


def aucs(df, objetivo, score="score"):
    def ev(metrica):
        return BinaryClassificationEvaluator(rawPredictionCol=score, labelCol=objetivo, metricName=metrica).evaluate(df)
    return {"auc_roc": round(ev("areaUnderROC"), 4), "auc_pr": round(ev("areaUnderPR"), 4)}


def grilla(parametros):
    claves = list(parametros)
    return [dict(zip(claves, v)) for v in itertools.product(*(parametros[k] for k in claves))]


def clasificador(tipo, objetivo, params):
    comunes = dict(featuresCol="vector", labelCol=objetivo, weightCol="peso", maxBins=MAX_BINS, seed=MF["semilla"])
    if tipo == "random_forest":
        return RandomForestClassifier(subsamplingRate=0.8, **comunes, **params)
    return GBTClassifier(**comunes, **params)


busqueda, ELEGIDOS = [], {}
for objetivo in OBJETIVOS:
    entrenamiento, peso = con_peso(train_v, objetivo)
    for tipo, parametros in (("random_forest", MF["grilla_rf"]), ("gbt", MF["grilla_gbt"])):
        for params in grilla(parametros):
            inicio = datetime.now(UTC)
            ajustado = clasificador(tipo, objetivo, params).fit(entrenamiento)
            segundos = (datetime.now(UTC) - inicio).total_seconds()
            pv = (con_evitable(con_score(ajustado.transform(valid_v)))
                  .select("score", objetivo, "es_latam", "pais", f"evit_{objetivo}").cache())
            lb = pv.where(ES_LATAM_BR)
            u, b = umbral_por_valor(curva(lb, objetivo, "score"), objetivo)
            fila = {"objetivo": objetivo, "tipo": tipo, "params": json.dumps(params), "segundos_entrenamiento": round(segundos),
                    "valid_latam_br_beneficio_usd": round(b), "valid_latam_br_umbral": u,
                    **{f"valid_total_{k}": v for k, v in aucs(pv, objetivo).items()},
                    **{f"valid_latam_br_{k}": v for k, v in aucs(lb, objetivo).items()}}
            pv.unpersist()
            busqueda.append(fila)
            print(fila)
            clave = (fila["valid_latam_br_beneficio_usd"], fila["valid_latam_br_auc_pr"])
            if objetivo not in ELEGIDOS or clave > ELEGIDOS[objetivo]["clave"]:
                ELEGIDOS[objetivo] = {"fila": fila, "clasificador": ajustado, "peso": peso, "clave": clave}

tabla_busqueda = pd.DataFrame(busqueda)
tabla_busqueda.to_csv(EVID / "busqueda_modelo_final.csv", index=False)
tabla_busqueda

## 3. Modelos finales

El modelo guardado es un solo `PipelineModel`: la preparación ajustada con train más el clasificador elegido. Así,
quien lo use (lote mensual o flujo) solo necesita las columnas de `gold/features_salida` y llamar a `transform`.

In [ ]:
MODELOS = {}
for objetivo, elegido in ELEGIDOS.items():
    modelo = PipelineModel(stages=[*preparacion.stages, elegido["clasificador"]])
    modelo.write().overwrite().save(f"{MODELOS_DIR}/final_{objetivo}")
    MODELOS[objetivo] = modelo
    print(f"{objetivo}: {elegido['fila']['tipo']} {elegido['fila']['params']} -> {MODELOS_DIR}/final_{objetivo}")

## 4. Referencias

- **Nunca alertar:** no hay sistema; todas las demoras llegan sin aviso.
- **Tasa histórica:** proporción de salidas demoradas por (país, origen, hora local) en train, con al menos
  `min_vuelos_tasa_hist` vuelos; si no, la tasa general de train. Se calcula aquí para las dos alertas.
- **Regresión logística base** (notebook 10), solo para la ámbar porque se entrenó con `y_demora15`.

In [ ]:
CLAVE_TASA = ["pais", "origen", "hora_local"]
MIN_TASA = cfg["features"]["min_vuelos_tasa_hist"]
tasas = (
    train_total.groupBy(*CLAVE_TASA)
    .agg(F.count("*").alias("_n"), F.avg("y_demora15").alias("_t15"), F.avg("y_demora60").alias("_t60"))
    .where(F.col("_n") >= MIN_TASA).drop("_n")
)
general = train_total.agg(F.avg("y_demora15").alias("g15"), F.avg("y_demora60").alias("g60")).first()


def con_referencia(df):
    return (
        df.join(F.broadcast(tasas), CLAVE_TASA, "left")
        .withColumn("ref_y_demora15", F.coalesce("_t15", F.lit(general.g15)))
        .withColumn("ref_y_demora60", F.coalesce("_t60", F.lit(general.g60)))
        .drop("_t15", "_t60")
    )


def puntuar(df, modelo, nombre):
    """Aplica un PipelineModel y deja solo la probabilidad de demora con el nombre pedido."""
    salida = modelo.transform(df).withColumn(nombre, vector_to_array("probability")[1])
    return salida.drop(*[c for c in salida.columns if c not in df.columns and c != nombre])


modelo_base = PipelineModel.load(f"{MODELOS_DIR}/modelo_base")


def todas_las_puntuaciones(df):
    df = con_referencia(df)
    df = puntuar(df, modelo_base, "base_y_demora15")
    for objetivo, modelo in MODELOS.items():
        df = puntuar(df, modelo, f"final_{objetivo}")
    return con_evitable(df)


valid_p = todas_las_puntuaciones(datos.where(F.col("particion") == "valid")).cache()
test_p = todas_las_puntuaciones(test).cache()
print(f"valid: {valid_p.count():,} · test: {test_p.count():,}")

## 5. Umbral por valor de negocio (elegido en valid, LATAM Brasil)

El umbral es la regla de decisión del CCO: **reprogramar con anticipación si la probabilidad ≥ umbral**. Se elige el
que deja más dólares en valid con el escenario y la fracción de `modelo_final`. Cada puntaje (referencia, base y
final) recibe su propio umbral con la misma regla, así se comparan en igualdad de condiciones.

La tabla de sensibilidad repite la elección con los tres escenarios de Westminster (bajo, base y alto) y con cada
fracción capturada *f*, porque *f* no tiene fuente. Si un umbral solo conviene con *f* alto, el CCO debe saberlo.

In [ ]:
PUNTAJES = {"y_demora15": ["ref_y_demora15", "base_y_demora15", "final_y_demora15"],
            "y_demora60": ["ref_y_demora60", "final_y_demora60"]}
valid_lb = valid_p.where(ES_LATAM_BR).cache()
dias_valid = valid_lb.select("fecha_local").distinct().count()
UMBRALES, sensibilidad = {}, []
for objetivo, puntajes in PUNTAJES.items():
    for score in puntajes:
        for esc in VN["escenarios"]:
            c = curva(con_evitable(valid_lb, esc), objetivo, score)
            for f in FRACCIONES:
                u, b = umbral_por_valor(c, objetivo, f, esc)
                fila = c.loc[min(int(round(u * TRAMOS)), TRAMOS - 1)] if u <= 1 else None
                alertas = 0 if fila is None else fila.alertas
                sensibilidad.append({
                    "objetivo": objetivo, "puntaje": score, "escenario": esc, "fraccion_capturada": f, "umbral": u,
                    "beneficio_valid_usd": round(b), "alertas_por_dia_valid": round(alertas / max(dias_valid, 1), 1),
                    "precision_valid": 0.0 if fila is None else round(fila.vp / max(fila.alertas, 1), 4),
                    "recall_valid": 0.0 if fila is None else round(fila.vp / max(c.pos.sum(), 1), 4),
                })
                if esc == ESC and f == F_SEL:
                    UMBRALES[score] = u
tabla_sensibilidad = pd.DataFrame(sensibilidad)
tabla_sensibilidad.to_csv(EVID / "sensibilidad_umbral.csv", index=False)
print("Umbrales publicados:", UMBRALES)
tabla_sensibilidad[tabla_sensibilidad.puntaje.str.startswith("final")]

## 6. Evaluación en test (2025)

Métricas por segmento: total, por país, LATAM Brasil, sus dos hubs (Guarulhos `SBGR` y Congonhas `SBSP`) frente al
resto, y las otras aerolíneas de Brasil (Azul y Gol), para ver si el modelo trata parejo a cada grupo.

- **AUC-ROC / AUC-PR:** qué tan bien ordena los vuelos por riesgo (la base del AUC-PR es la tasa de demora).
- **Precisión / recall / F1** al umbral publicado.
- **Beneficio (US$):** costo evitable × fracción de selección − costo de reprogramar las alertas, en el escenario de
  selección. "Nunca alertar" vale 0 por definición, así que un beneficio positivo significa que reprogramar con esa
  lista es mejor que no hacer nada.
- **Punto de equilibrio *f\***:** fracción del costo evitable que hay que capturar para que la lista se pague.
  Cuanto más bajo, mejor.

Los costos están ponderados por la flota de LATAM. En los segmentos de EE. UU. se informan solo para comparar.

In [ ]:
HUBS = MF["hubs"]
SEGMENTOS = {
    "Total": F.lit(True),
    "EE. UU.": F.col("pais") == "US",
    "Brasil (todas las aerolíneas)": F.col("pais") == "BR",
    "LATAM Brasil": ES_LATAM_BR,
    **{f"LATAM Brasil · {h}": ES_LATAM_BR & (F.col("origen") == h) for h in HUBS},
    "LATAM Brasil · resto": ES_LATAM_BR & ~F.col("origen").isin(HUBS),
    **{f"Brasil · {cfg['competidores'][c]}": (F.col("pais") == "BR") & (F.col("aerolinea") == c) for c in cfg["competidores"]},
}


def metricas(df, objetivo, score, umbral):
    n = df.count()
    if n == 0 or df.select(objetivo).distinct().count() < 2:
        return {"vuelos": n}
    c = df.select(
        F.sum(((F.col(score) >= umbral) & (F.col(objetivo) == 1)).cast("int")).alias("vp"),
        F.sum(((F.col(score) >= umbral) & (F.col(objetivo) == 0)).cast("int")).alias("fp"),
        F.sum(((F.col(score) < umbral) & (F.col(objetivo) == 1)).cast("int")).alias("fn"),
        F.avg(objetivo).alias("tasa"),
        F.sum(F.when(F.col(score) >= umbral, F.col(f"evit_{objetivo}")).otherwise(0.0)).alias("evit"),
    ).first()
    precision = c.vp / (c.vp + c.fp) if (c.vp + c.fp) else 0.0
    recall = c.vp / (c.vp + c.fn) if (c.vp + c.fn) else 0.0
    alertas = c.vp + c.fp
    costo_rep = alertas * costo_reprogramar(PRODUCTO[objetivo], ESC)
    return {
        "vuelos": n, "tasa_demora": round(c.tasa, 4), **aucs(df, objetivo, score),
        "precision": round(precision, 4), "recall": round(recall, 4),
        "f1": round(2 * precision * recall / (precision + recall), 4) if precision + recall else 0.0,
        "pct_alertado": round((c.vp + c.fp) / n, 4), "umbral": round(umbral, 3),
        "vp": c.vp, "fp": c.fp, "fn": c.fn, "vn": n - c.vp - c.fp - c.fn,
        "costo_evitable_usd": round(c.evit or 0.0), "costo_reprogramar_usd": round(costo_rep),
        "beneficio_usd": round(F_SEL * (c.evit or 0.0) - costo_rep),
        "f_equilibrio": round(costo_rep / c.evit, 4) if c.evit else None,
    }


filas = []
for segmento, condicion in SEGMENTOS.items():
    parte = test_p.where(condicion)
    for objetivo, puntajes in PUNTAJES.items():
        for score in puntajes:
            filas.append({"segmento": segmento, "objetivo": objetivo, "puntaje": score,
                          **metricas(parte, objetivo, score, UMBRALES[score])})
tabla_test = pd.DataFrame(filas)
tabla_test.to_csv(EVID / "metricas_modelo_final.csv", index=False)
tabla_test[["segmento", "objetivo", "puntaje", "vuelos", "tasa_demora", "auc_roc", "auc_pr", "precision", "recall",
            "f1", "beneficio_usd", "f_equilibrio"]]

## 7. ¿Le deja más dinero a LATAM que las referencias? (test 2025, LATAM Brasil)

El modelo se recomienda solo si su beneficio supera al de la tabla de tasas por aeropuerto y hora y al de no hacer
nada (0 US$). El notebook 15 repite la comparación con el mismo número de alertas y con intervalos de confianza.

In [ ]:
lb = tabla_test[tabla_test.segmento == "LATAM Brasil"].set_index(["objetivo", "puntaje"])
resumen_lb = lb[["precision", "recall", "beneficio_usd", "f_equilibrio", "auc_roc", "auc_pr"]]
for objetivo in OBJETIVOS:
    final, ref = lb.loc[(objetivo, f"final_{objetivo}")], lb.loc[(objetivo, f"ref_{objetivo}")]
    supera = final.beneficio_usd > max(ref.beneficio_usd, 0)
    print(f"{objetivo}: beneficio 2025 modelo US$ {final.beneficio_usd:,.0f} vs tabla de tasas US$ {ref.beneficio_usd:,.0f} "
          f"vs nunca alertar US$ 0 · punto de equilibrio {final.f_equilibrio} vs {ref.f_equilibrio} -> "
          f"{'SUPERA a la tabla y a no hacer nada' if supera else 'NO supera (decirlo en el informe)'}")
resumen_lb

In [ ]:
# Matriz de confusión de LATAM Brasil en test, al umbral publicado
matrices = []
for objetivo in OBJETIVOS:
    f = lb.loc[(objetivo, f"final_{objetivo}")]
    matrices.append(pd.DataFrame(
        [[int(f.vn), int(f.fp)], [int(f.fn), int(f.vp)]],
        index=pd.Index(["real: a tiempo", f"real: demora ({objetivo})"]),
        columns=["predicho: sin alerta", "predicho: alerta"],
    ))
    print(f"\n{objetivo} (umbral {f.umbral})")
    print(matrices[-1])

## 8. Cierre del gobierno de EE. UU. (1-oct a 15-nov de 2025)

Durante el cierre faltaron controladores de tránsito aéreo y la FAA recortó vuelos: es un periodo atípico que no
existe en train. Se reporta EE. UU. con y sin esas semanas para ver cuánto pesa en las métricas.

In [ ]:
desde_cierre, hasta_cierre = MF["cierre_gobierno_eeuu"]
en_cierre = F.col("fecha_local").between(F.lit(desde_cierre).cast("date"), F.lit(hasta_cierre).cast("date"))
filas_cierre = []
for nombre, condicion in {"EE. UU. todo 2025": F.lit(True), "EE. UU. sin el cierre": ~en_cierre,
                          "EE. UU. solo el cierre": en_cierre}.items():
    parte = test_p.where((F.col("pais") == "US") & condicion)
    filas_cierre.append({"periodo": nombre, **metricas(parte, "y_demora15", "final_y_demora15",
                                                       UMBRALES["final_y_demora15"])})
tabla_cierre = pd.DataFrame(filas_cierre)
tabla_cierre.to_csv(EVID / "metricas_cierre_gobierno_eeuu.csv", index=False)
tabla_cierre[["periodo", "vuelos", "tasa_demora", "auc_roc", "auc_pr", "precision", "recall"]]

## 9. Importancia de variables

`featureImportances` dice cuánto usó el bosque cada variable para cortar, no cuánto *causa* la demora. Dos
limitaciones: favorece a las variables con muchos valores posibles y, si dos variables están correlacionadas (por
ejemplo `red_demora_media` y `red_frac_atrasadas`), reparte la importancia entre ellas.

In [ ]:
AZUL, GRIS, TINTA = "#2a78d6", "#c9c8c2", "#1f1f1f"
importancias = []
fig, ejes = plt.subplots(1, len(OBJETIVOS), figsize=(12, 5), sharex=False)
for eje, objetivo in zip(ejes, OBJETIVOS):
    valores = MODELOS[objetivo].stages[-1].featureImportances.toArray()
    tabla = pd.DataFrame({"variable": NOMBRES_VECTOR, "importancia": valores}).sort_values("importancia", ascending=False)
    tabla["objetivo"] = objetivo
    importancias.append(tabla)
    top = tabla.head(12).iloc[::-1]
    eje.barh(top.variable, top.importancia, color=AZUL)
    eje.set_title(f"{objetivo}: ¿qué variables usa más el modelo?", fontsize=10, color=TINTA, loc="left")
    eje.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(EVID / "importancia_variables.png", dpi=130, bbox_inches="tight")
pd.concat(importancias).to_csv(EVID / "importancia_variables.csv", index=False)
plt.show()

## 10. Alternativa descartada: predecir minutos

Un `GBTRegressor` sobre la demora en minutos (recortada a [−30, 300]) frente a predecir siempre la media de train.
Si el R² es bajo, conviene más clasificar (¿pasará de 15 o 60 min?) que estimar los minutos exactos.

In [ ]:
recorte = F.least(F.greatest(F.col("demora_salida_min").cast("double"), F.lit(-30.0)), F.lit(300.0))
tr_min = train_v.withColumn("minutos", recorte).where(F.col("minutos").isNotNull())
media = tr_min.agg(F.avg("minutos")).first()[0]
regresor = GBTRegressor(featuresCol="vector", labelCol="minutos", maxIter=40, maxDepth=5, maxBins=MAX_BINS,
                        seed=MF["semilla"]).fit(tr_min)
te_min = preparacion.transform(test.where(ES_LATAM_BR)).withColumn("minutos", recorte).where(F.col("minutos").isNotNull())
pred_min = regresor.transform(te_min)
trivial = te_min.withColumn("prediction", F.lit(media))
filas_reg = []
for nombre, df in (("gbt_regresion", pred_min), ("media_train", trivial)):
    filas_reg.append({"modelo": nombre, **{m: round(RegressionEvaluator(labelCol="minutos", predictionCol="prediction",
                                                                       metricName=m).evaluate(df), 3)
                                           for m in ("rmse", "mae", "r2")}})
tabla_reg = pd.DataFrame(filas_reg)
tabla_reg.to_csv(EVID / "alternativa_regresion_minutos.csv", index=False)
tabla_reg

## 11. Umbrales publicados y resumen

Los umbrales se guardan junto a los modelos en gold: el lote mensual (notebook 13) y la capa de velocidad
(notebook 17) los leen de ahí, así las dos capas alertan con la misma regla.

In [ ]:
publicados = [(objetivo, float(UMBRALES[f"final_{objetivo}"]), VERSION, ESC, F_SEL, ELEGIDOS[objetivo]["fila"]["tipo"],
               ELEGIDOS[objetivo]["fila"]["params"]) for objetivo in OBJETIVOS]
(
    spark.createDataFrame(publicados, "objetivo string, umbral double, version string, escenario string, "
                                      "fraccion_capturada double, "
                                      "tipo string, params string")
    .coalesce(1).write.mode("overwrite").json(f"{MODELOS_DIR}/umbrales_final")
)
resultado = {
    "ejecucion_utc": datetime.now(UTC).isoformat(timespec="seconds"), "version": VERSION,
    "muestra_train": fr_train, "muestra_valid": fr_valid, "variables_numericas": NUM_USADAS,
    "variables_categoricas": CATEGORICAS, "escenario_seleccion": ESC, "fraccion_seleccion": F_SEL,
    "factor_eur2014_a_usd": FACTOR_USD, "peso_avion": PESO_AVION, "umbrales": UMBRALES,
    "elegidos": {o: e["fila"] for o, e in ELEGIDOS.items()},
    "latam_brasil_test": tabla_test[tabla_test.segmento == "LATAM Brasil"].to_dict(orient="records"),
}
(EVID / "modelo_final.json").write_text(json.dumps(resultado, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
print("Umbrales publicados:", publicados)

In [ ]:
spark.stop()